In [3]:
!python -V

Python 3.11.5


In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

In [4]:
from dotenv import load_dotenv
load_dotenv()

import os
print(os.environ.get("MLFLOW_S3_ENDPOINT_URL"))
print(os.environ.get("AWS_ACCESS_KEY_ID"))
print(os.environ.get("AWS_SECRET_ACCESS_KEY"))
print(os.environ.get("AWS_DEFAULT_REGION"))

https://t3.storage.dev
tid_eFNaNVZsjpfZKBSMVaXQmrGFKlanjPVSuxuyoXXaeXeKmduDwo
tsec__7Ga_MFdDGPIqD4wCgsyd6QMR75W_9DVHziGBsTXOSMsIGB_SGBg49sDoNt0Gs4tmxxfyi
auto


In [5]:
import mlflow

# mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("nyc-taxi-experiment")

/Users/amruthakaruturi/Library/Caches/pypoetry/virtualenvs/server-pa4cgUtV-py3.11/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='s3://nyc-taxi-pred-models/1', creation_time=1778506704923, experiment_id='1', last_update_time=1778506704923, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
# Try using fastparquet if you have it installed
# df = pd.read_parquet('./data/green_tripdata_2021-01.parquet', engine='fastparquet')
def read_dataframe(filename):

    df = pd.read_parquet(filename, engine="fastparquet")

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df["PU_DO"] = df["PULocationID"] + '_' + df["DOLocationID"]
    
    return df

In [8]:
!pip install fastparquet

  Using cached fsspec-2026.4.0-py3-none-any.whl.metadata (10 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 686.0/686.0 kB 5.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.1 MB/s  0:00:00
Using cached fsspec-2026.4.0-py3-none-any.whl (203 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [8]:
categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [9]:

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [10]:
import xgboost as xgb 

In [12]:
from pathlib import Path

models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [13]:
mlflow.xgboost.autolog(disable=True)

In [14]:
with mlflow.start_run():
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        "max_depth": 20,
        "learning_rate": 0.666730261582146,
        "reg_alpha": 0.06229964677766508,
        "reg_lambda": 0.02599517340753995,
        "min_child_weight": 1.4071004781961702,
        "seed": 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params = best_params,
        dtrain = train,
        num_boost_round = 100,
        evals = [(valid, "validation")],
        early_stopping_rounds = 5
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")
    mlflow.xgboost.log_model(booster, artifact_path = "models_mlflow")

2026/05/11 08:56:45 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



[0]	validation-rmse:7.69164
[1]	validation-rmse:6.85656
[2]	validation-rmse:6.68397
[3]	validation-rmse:6.64277
[4]	validation-rmse:6.63128
[5]	validation-rmse:6.62306
[6]	validation-rmse:6.61656
[7]	validation-rmse:6.60618
[8]	validation-rmse:6.60010
[9]	validation-rmse:6.59891
[10]	validation-rmse:6.59252
[11]	validation-rmse:6.58741
[12]	validation-rmse:6.58384
[13]	validation-rmse:6.58004
[14]	validation-rmse:6.57691
[15]	validation-rmse:6.57066
[16]	validation-rmse:6.56502
[17]	validation-rmse:6.55772
[18]	validation-rmse:6.55468
[19]	validation-rmse:6.54978
[20]	validation-rmse:6.54608
[21]	validation-rmse:6.54084
[22]	validation-rmse:6.53772
[23]	validation-rmse:6.53244
[24]	validation-rmse:6.52987
[25]	validation-rmse:6.52680
[26]	validation-rmse:6.52565
[27]	validation-rmse:6.52118
[28]	validation-rmse:6.51778
[29]	validation-rmse:6.51449
[30]	validation-rmse:6.51198
[31]	validation-rmse:6.51025
[32]	validation-rmse:6.50360
[33]	validation-rmse:6.50226
[34]	validation-rmse:6.5

2026/05/11 08:56:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run capable-robin-22 at: http://127.0.0.1:5001/#/experiments/1/runs/2b59bba25cad4b2bab5b20b1ccc1ab37
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/1


In [24]:
import boto3, os

s3 = boto3.client(
    "s3",
    endpoint_url=os.environ["MLFLOW_S3_ENDPOINT_URL"],
    aws_access_key_id=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_access_key=os.environ["AWS_SECRET_ACCESS_KEY"]
)

# List all buckets — see what name actually shows up
buckets = s3.list_buckets()
print([b["Name"] for b in buckets["Buckets"]])

# Try accessing it directly
s3.head_bucket(Bucket="nyc-taxi-pred-models")
print("Bucket accessible!")

['full-stack-rag', 'mlflow-models', 'nyc-taxi-pred-models']
Bucket accessible!


In [28]:
client = mlflow.tracking.MlflowClient()
exp = client.get_experiment_by_name("nyc-taxi-experiment")
print(exp.artifact_location)

s3://nyc-taxi-pred-models/1
